In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
import importlib
import hashlib
import sys
import types
from unittest.mock import patch
from pathlib import Path

In [2]:
def normalize_fitzpatrick(series, *, missing_codes=()):
    roman_map = {
        "i": "1",
        "ii": "2",
        "iii": "3",
        "iv": "4",
        "v": "5",
        "vi": "6",
    }

    text = (
        series.astype("string")
        .str.strip()
        .str.lower()
        .replace("", pd.NA)
        .replace(roman_map)
    )

    numeric = pd.to_numeric(text, errors="raise")
    numeric = numeric.mask(numeric.isin(missing_codes))

    invalid = numeric.notna() & ~numeric.isin([1, 2, 3, 4, 5, 6])

    if invalid.any():
        raise ValueError(
            "Unexpected Fitzpatrick values: "
            f"{series.loc[invalid].dropna().unique().tolist()}"
        )

    return numeric.astype("Int64")

In [3]:
CLASSIFICATION_MAP = {
    "benign": "benign",
    "malignant": "malignant",
    "premalignant": "premalignant",
    "non-neoplastic": "non_neoplastic",
    "non_neoplastic": "non_neoplastic",
    "indeterminate": "indeterminate",
    "other": "other",
}

# Specific locations:
# ABDOMEN, ARM, BACK, CHEST, EAR, FACE, FOOT, FOREARM,
# HAND, LIP, NECK, NOSE, SCALP, THIGH, LOWER_LEG,
# SHOULDER, ELBOW, WRIST, ANKLE, KNEE, FLANK,
# HIP, BUTTOCK, GROIN, PUBIC_REGION, AXILLA

# Broad locations retained when no finer detail is available:
# HEAD_NECK, UPPER_EXTREMITY, LOWER_EXTREMITY,
# ANTERIOR_TORSO, POSTERIOR_TORSO, LATERAL_TORSO,
# ORAL_GENITAL, PALMS_SOLES

GENERAL_SITE_MAP = {
    "anterior torso": "ANTERIOR_TORSO",
    "posterior torso": "POSTERIOR_TORSO",
    "lateral torso": "LATERAL_TORSO",
    "head/neck": "HEAD_NECK",
    "upper extremity": "UPPER_EXTREMITY",
    "lower extremity": "LOWER_EXTREMITY",
    "oral/genital": "ORAL_GENITAL",
    "palms/soles": "PALMS_SOLES",
}

def normalize_site(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).replace("Â", "")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text if text else pd.NA


MRA_SITE_MAP = {
    # ABDOMEN
    'abdomen': 'ABDOMEN',
    'central upper abdomen': 'ABDOMEN',
    'hypogastric region': 'ABDOMEN',
    'l abdomen': 'ABDOMEN',
    'l lower abdomen': 'ABDOMEN',
    'l upper abdomen': 'ABDOMEN',
    'left abdomen': 'ABDOMEN',
    'lower abdomen': 'ABDOMEN',
    'mid abdomen': 'ABDOMEN',
    'r abdomen': 'ABDOMEN',
    'r upper abdomen': 'ABDOMEN',
    'right abdomen': 'ABDOMEN',
    'right upper abdomen': 'ABDOMEN',
    'umbilicus': 'ABDOMEN',
    'upper mid abdomen': 'ABDOMEN',

    # ANKLE
    'l medial ankle': 'ANKLE',
    'l posterior ankle': 'ANKLE',
    'r medial malleolus': 'ANKLE',
    'right ankle': 'ANKLE',
    'right lateral ankle': 'ANKLE',

    # ARM
    'arm': 'ARM',
    'l arm': 'ARM',
    'l deltoid': 'ARM',
    'l inferior upper arm': 'ARM',
    'l inner upper arm': 'ARM',
    'l upper arm': 'ARM',
    'left arm': 'ARM',
    'left arm lateral': 'ARM',
    'left arm medial': 'ARM',
    'left upper arm': 'ARM',
    'r arm': 'ARM',
    'r deltoid': 'ARM',
    'r distal lateral upper arm': 'ARM',
    'r posterior arm': 'ARM',
    'r upper arm': 'ARM',
    'right anterior upper arm': 'ARM',
    'right arm': 'ARM',
    'right medial arm': 'ARM',
    'right superior upper arm': 'ARM',
    'right upper arm': 'ARM',

    # AXILLA
    'l axilla': 'AXILLA',
    'r axilla': 'AXILLA',

    # BACK
    'back': 'BACK',
    'central lower back': 'BACK',
    'central upper back': 'BACK',
    'l back': 'BACK',
    'l lower back': 'BACK',
    'l mid back': 'BACK',
    'l upper back': 'BACK',
    'l upper back (this is mislabeled on epic as r upper back)': 'BACK',
    'left back': 'BACK',
    'left central back': 'BACK',
    'left lower back': 'BACK',
    'left mid back': 'BACK',
    'left scapula': 'BACK',
    'left upper back': 'BACK',
    'low back': 'BACK',
    'low mid back': 'BACK',
    'lower back': 'BACK',
    'lower mid back': 'BACK',
    'mid back': 'BACK',
    'mid central back': 'BACK',
    'mid low back': 'BACK',
    'mid lower back': 'BACK',
    'mid upper back': 'BACK',
    'r back': 'BACK',
    'r lateral back': 'BACK',
    'r lower back': 'BACK',
    'r mid back': 'BACK',
    'r upper back': 'BACK',
    'right back': 'BACK',
    'right lateral mid back': 'BACK',
    'right lower back': 'BACK',
    'right mid back': 'BACK',
    'right mid-back': 'BACK',
    'right upper back': 'BACK',
    'superior l upper back': 'BACK',
    'upper back': 'BACK',
    'upper mid back': 'BACK',

    # BUTTOCK
    'left buttock': 'BUTTOCK',
    'left upper buttock': 'BUTTOCK',
    'right buttock': 'BUTTOCK',
    'right buttox': 'BUTTOCK',

    # CHEST
    'central chest': 'CHEST',
    'chest': 'CHEST',
    'l breast': 'CHEST',
    'l chest': 'CHEST',
    'l chest lateral': 'CHEST',
    'l chest medial': 'CHEST',
    'l clavicle': 'CHEST',
    'l inframammary': 'CHEST',
    'l ribcage': 'CHEST',
    'l upper chest': 'CHEST',
    'left breast': 'CHEST',
    'left chest': 'CHEST',
    'left clavicle': 'CHEST',
    'left infra mammary': 'CHEST',
    'mid chest': 'CHEST',
    'r chest': 'CHEST',
    'r clavicle': 'CHEST',
    'r low chest': 'CHEST',
    'r lower chest': 'CHEST',
    'r medial chest': 'CHEST',
    'r upper chest': 'CHEST',
    'right chest': 'CHEST',
    'right clavicle': 'CHEST',
    'upper chest': 'CHEST',
    'upper mid r chest': 'CHEST',

    # EAR
    'l antihelix': 'EAR',
    'l ear': 'EAR',
    'l ear scapha': 'EAR',
    'l helix': 'EAR',
    'l posterior helix': 'EAR',
    'l superior helix': 'EAR',
    'l superior helix of ear': 'EAR',
    'left ear': 'EAR',
    'left helix': 'EAR',
    'left lobule of ear': 'EAR',
    'left posterior helix': 'EAR',
    'r antihelix': 'EAR',
    'r antitragus': 'EAR',
    'r conchal bowl': 'EAR',
    'r posterior helix': 'EAR',
    'r superior helix': 'EAR',
    'right ear': 'EAR',
    'right helix': 'EAR',
    'right posterior helix': 'EAR',
    'right superior helix': 'EAR',
    'right superior helix of ear': 'EAR',

    # ELBOW
    'l elbow': 'ELBOW',
    'left elbow': 'ELBOW',
    'r elbow': 'ELBOW',
    'r elbow joint': 'ELBOW',
    'right elbow': 'ELBOW',

    # FACE
    'cheek': 'FACE',
    'chin': 'FACE',
    'forehead': 'FACE',
    'l cheek': 'FACE',
    'l eyebrow': 'FACE',
    'l forehead': 'FACE',
    'l jaw': 'FACE',
    'l jawline': 'FACE',
    'l lateral cheek': 'FACE',
    'l lower eyelid': 'FACE',
    'l lower medial cheek': 'FACE',
    'l malar cheek': 'FACE',
    'l mandible': 'FACE',
    'l medial cheek': 'FACE',
    'l medial eyebrow': 'FACE',
    'l temple': 'FACE',
    'left cheek': 'FACE',
    'left eyebrow': 'FACE',
    'left forehead': 'FACE',
    'left jawline': 'FACE',
    'left lateral cheek': 'FACE',
    'left lateral jawline': 'FACE',
    'left temple': 'FACE',
    'mid forehead': 'FACE',
    'r cheek': 'FACE',
    'r eyebrow': 'FACE',
    'r eyelid': 'FACE',
    'r forehead': 'FACE',
    'r jaw': 'FACE',
    'r lateral cheek': 'FACE',
    'r lateral eyelid': 'FACE',
    'r lower cheek': 'FACE',
    'r lower eyelid': 'FACE',
    'r medial cheek': 'FACE',
    'r medial eyebrow': 'FACE',
    'r medial lower eyelid': 'FACE',
    'r preauricular cheek': 'FACE',
    'r side burn': 'FACE',
    'r sideburn': 'FACE',
    'r temple': 'FACE',
    'r upper eyelid': 'FACE',
    'right cheek': 'FACE',
    'right chin': 'FACE',
    'right forehead': 'FACE',
    'right jaw': 'FACE',
    'right lateral canthus': 'FACE',
    'right lateral cheek': 'FACE',
    'right lower eyelid': 'FACE',
    'right medial cheek': 'FACE',
    'right temple': 'FACE',
    'right upper eyelid': 'FACE',
    'submental jaw': 'FACE',

    # FLANK
    'l flank': 'FLANK',
    'l flank inferior': 'FLANK',
    'l flank superior': 'FLANK',
    'left lower flank': 'FLANK',
    'r flank': 'FLANK',
    'right flank': 'FLANK',

    # FOOT
    'l dorsal 2nd toe': 'FOOT',
    'l dorsal foot': 'FOOT',
    'l foot': 'FOOT',
    'l medial plantar heel': 'FOOT',
    'left dorsal foot': 'FOOT',
    'left heel': 'FOOT',
    'left mid distal dorsal foot': 'FOOT',
    'left mid plantar heel': 'FOOT',
    'r 4th dorsal toe': 'FOOT',
    'r dorsal foot': 'FOOT',
    'r foot': 'FOOT',
    'r heel': 'FOOT',
    'r lateral foot': 'FOOT',
    'right 3rd intertarsal region': 'FOOT',
    'right plantar arch': 'FOOT',

    # FOREARM
    'l dorsal forearm': 'FOREARM',
    'l forearm': 'FOREARM',
    'l proximal dorsal forearm': 'FOREARM',
    'left forearm': 'FOREARM',
    'r forearm': 'FOREARM',
    'r medial forearm': 'FOREARM',
    'right distal forearm': 'FOREARM',
    'right forearm': 'FOREARM',
    'right proximal forearm': 'FOREARM',

    # GROIN
    'l inguinal crease': 'GROIN',
    'left groin': 'GROIN',
    'right inguinal fold': 'GROIN',

    # HAND
    'l 2nd finger': 'HAND',
    'l 5th finger': 'HAND',
    'l dorsal hand': 'HAND',
    'l hand': 'HAND',
    'l hand radial': 'HAND',
    'l hand ulnar': 'HAND',
    'l middle finger': 'HAND',
    'left 4th finger': 'HAND',
    'left dorsal hand': 'HAND',
    'r 3rd finger': 'HAND',
    'r dorsal hand': 'HAND',
    'r index finger': 'HAND',
    'r thumb base': 'HAND',
    'right dorsal hand': 'HAND',
    'right hand': 'HAND',

    # HEAD_NECK
    'l infra-auricular': 'HEAD_NECK',
    'l postauricular': 'HEAD_NECK',
    'l preauricular': 'HEAD_NECK',
    'left post auricular': 'HEAD_NECK',
    'left postauricular': 'HEAD_NECK',
    'left postauricular area': 'HEAD_NECK',
    'left pre-auricular': 'HEAD_NECK',
    'r preauricular': 'HEAD_NECK',
    'r retro auricular': 'HEAD_NECK',
    'right postauricular': 'HEAD_NECK',
    'right pre-auricular': 'HEAD_NECK',

    # HIP
    'l hip': 'HIP',

    # KNEE
    'l knee': 'KNEE',
    'l lateral knee': 'KNEE',
    'l popliteal': 'KNEE',
    'left knee': 'KNEE',
    'r lateral knee': 'KNEE',
    'r popliteal fossa': 'KNEE',
    'right knee': 'KNEE',
    'right medial knee': 'KNEE',

    # LIP
    'l lower lip': 'LIP',
    'l upper vermilion border': 'LIP',
    'left cutaneous lip': 'LIP',
    'left lower cutaneous lip': 'LIP',
    'left lower vermilion border of lip': 'LIP',
    'left upper cutaneous lip': 'LIP',
    'left upper lip': 'LIP',
    'lower lip': 'LIP',
    'lower vermilion lip': 'LIP',
    'mid upper cutaneous lip': 'LIP',
    'philtrum': 'LIP',
    'r upper lip': 'LIP',
    'upper middle lip': 'LIP',

    # LOWER_EXTREMITY
    'l leg': 'LOWER_EXTREMITY',
    'left leg': 'LOWER_EXTREMITY',
    'left posterior leg': 'LOWER_EXTREMITY',
    'leg': 'LOWER_EXTREMITY',
    'r lateral leg': 'LOWER_EXTREMITY',
    'r leg': 'LOWER_EXTREMITY',
    'r lower lateral leg': 'LOWER_EXTREMITY',
    'r proximal leg melanoma scar': 'LOWER_EXTREMITY',
    'right anterior leg': 'LOWER_EXTREMITY',
    'right leg': 'LOWER_EXTREMITY',

    # LOWER_LEG
    'l anterior lower leg': 'LOWER_LEG',
    'l anterior shin': 'LOWER_LEG',
    'l calf': 'LOWER_LEG',
    'l lateral calf': 'LOWER_LEG',
    'l lower calf': 'LOWER_LEG',
    'l lower leg': 'LOWER_LEG',
    'l medial calf': 'LOWER_LEG',
    'l medial shin': 'LOWER_LEG',
    'l posterior calf': 'LOWER_LEG',
    'l shin': 'LOWER_LEG',
    'l upper calf': 'LOWER_LEG',
    'left anterior lower leg': 'LOWER_LEG',
    'left anterior shin': 'LOWER_LEG',
    'left calf': 'LOWER_LEG',
    'left lower leg': 'LOWER_LEG',
    'left posterior calf': 'LOWER_LEG',
    'left shin': 'LOWER_LEG',
    'r calf': 'LOWER_LEG',
    'r lateral calf': 'LOWER_LEG',
    'r lower leg': 'LOWER_LEG',
    'r lower leg inferior': 'LOWER_LEG',
    'r medial shin': 'LOWER_LEG',
    'r posterior calf': 'LOWER_LEG',
    'r shin': 'LOWER_LEG',
    'r upper post calf': 'LOWER_LEG',
    'right anterior shin': 'LOWER_LEG',
    'right calf': 'LOWER_LEG',
    'right lateral lower leg': 'LOWER_LEG',
    'right lower leg': 'LOWER_LEG',
    'right medial lower leg': 'LOWER_LEG',
    'right post calf': 'LOWER_LEG',
    'right posterior calf': 'LOWER_LEG',
    'right shin': 'LOWER_LEG',

    # NECK
    'l anterior neck': 'NECK',
    'l lateral neck': 'NECK',
    'l neck': 'NECK',
    'l posterior neck': 'NECK',
    'l root of neck': 'NECK',
    'left lateral neck': 'NECK',
    'left neck': 'NECK',
    'left posterior neck': 'NECK',
    'mid posterior neck': 'NECK',
    'neck': 'NECK',
    'posterior midline neck': 'NECK',
    'posterior neck': 'NECK',
    'r base of neck': 'NECK',
    'r lateral neck': 'NECK',
    'r neck': 'NECK',
    'r posterior neck': 'NECK',
    'right lateral neck': 'NECK',
    'right neck': 'NECK',

    # NOSE
    'l alar crease': 'NOSE',
    'l nasal ala': 'NOSE',
    'l nasal bridge': 'NOSE',
    'l nasal sidewall': 'NOSE',
    'l nasal supratip': 'NOSE',
    'l nasal tip': 'NOSE',
    'l nose': 'NOSE',
    'left alar crease (peri-nasal)': 'NOSE',
    'left alar crease of nose': 'NOSE',
    'left nasal ala': 'NOSE',
    'left nasal bridge': 'NOSE',
    'left nasal dorsum': 'NOSE',
    'left nasal supratip': 'NOSE',
    'mid nasal dorsum': 'NOSE',
    'mid nasal supra tip': 'NOSE',
    'mid nasal supratip': 'NOSE',
    'mid-nasal dorsum': 'NOSE',
    'nasal bridge': 'NOSE',
    'nasal dorsum': 'NOSE',
    'nasal tip': 'NOSE',
    'nose': 'NOSE',
    'r nasal ala': 'NOSE',
    'r nasal bridge': 'NOSE',
    'r nasal fold': 'NOSE',
    'r nasal sidewall': 'NOSE',
    'r nasal supratip': 'NOSE',
    'r nose tip': 'NOSE',
    'r supra nasal tip': 'NOSE',
    'right nasal ala': 'NOSE',
    'right nasal sidewall': 'NOSE',
    'right nasal supra tip': 'NOSE',
    'right nasal tip': 'NOSE',

    # PUBIC_REGION
    'l suprapubic': 'PUBIC_REGION',
    'mons pubis': 'PUBIC_REGION',
    'supra pubic skin': 'PUBIC_REGION',

    # SCALP
    'crown of scalp': 'SCALP',
    'frontal scalp': 'SCALP',
    'l crown of scalp': 'SCALP',
    'l frontal scalp': 'SCALP',
    'l occiput': 'SCALP',
    'l parietal scalp': 'SCALP',
    'left frontal scalp': 'SCALP',
    'left scalp vertex': 'SCALP',
    'left vertex scalp': 'SCALP',
    'mid l vertex scalp': 'SCALP',
    'mid scalp': 'SCALP',
    'mid vertex': 'SCALP',
    'mid vertex scalp': 'SCALP',
    'r crown of scalp': 'SCALP',
    'r frontal hairline': 'SCALP',
    'r frontal scalp': 'SCALP',
    'r mid vertex': 'SCALP',
    'r post auricular scalp': 'SCALP',
    'r vertex scalp': 'SCALP',
    'right anterior scalp': 'SCALP',
    'right crown of scalp': 'SCALP',
    'right frontal scalp': 'SCALP',
    'right vertex scalp': 'SCALP',
    'scalp': 'SCALP',

    # SHOULDER
    'l posterior shoulder': 'SHOULDER',
    'l shoulder': 'SHOULDER',
    'left anterior shoulder': 'SHOULDER',
    'left posterior shoulder': 'SHOULDER',
    'left shoulder': 'SHOULDER',
    'left shoulder anterior': 'SHOULDER',
    'left shoulder lateral': 'SHOULDER',
    'left shoulder posterior': 'SHOULDER',
    'r posterior shoulder': 'SHOULDER',
    'r shoulder': 'SHOULDER',
    'r upper shoulder': 'SHOULDER',
    'right shoulder': 'SHOULDER',
    'shoulder': 'SHOULDER',

    # THIGH
    'l anterior thigh': 'THIGH',
    'l lateral thigh': 'THIGH',
    'l medial thigh': 'THIGH',
    'l thigh': 'THIGH',
    'left anterior thigh': 'THIGH',
    'left inner thigh': 'THIGH',
    'left mid thigh': 'THIGH',
    'left posterior thigh': 'THIGH',
    'left thigh': 'THIGH',
    'left thigh distal': 'THIGH',
    'left thigh proximal': 'THIGH',
    'left upper thigh': 'THIGH',
    'r medial thigh': 'THIGH',
    'r posterior thigh': 'THIGH',
    'r thigh': 'THIGH',
    'right lateral thigh': 'THIGH',
    'right posterior thigh': 'THIGH',
    'right superomedial thigh': 'THIGH',
    'right thigh': 'THIGH',
    'thigh': 'THIGH',

    # WRIST
    'l wrist': 'WRIST',

    # Unknown or held for review; intentionally not assigned a site.
    'l melolabia fold': pd.NA,
    'r infirmary breast': pd.NA,
    'r pinky': pd.NA,
    'r4 digit': pd.NA,
    'rt inferior parietal': pd.NA,
    'rt superior parietal': pd.NA,
    'unclear': pd.NA,
}


In [4]:
# The canonical schema for our model
PROJECT_ROOT = Path(r"C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2")
META_DIR = Path(r"C:\Users\12148\OneDrive\Documents\Capstone\Datasets\metadata")
OUTPUT_DIR = PROJECT_ROOT / "manifests"
if not META_DIR.is_dir():
    raise FileNotFoundError(f"Metadata folder not found: {META_DIR}")

CANONICAL = ['image_ref', 'hash_code', 'image_type', 'file_name',  'image_id',  'lesion_id', 'patient_id', 'dataset_source', 'split', 'fitzpatrick', 'fitzpatrick_group', 'skin_tone', 'tone_provenance', 
             'sex', 'age', 'anatomical_site', 'size', 'disease_label', 'label_class', 'disease_classification', 'pathology_confirmation', 'pathology',
             'race', 'ethnicity', 'family_history', 'personal_history', 'cancer_history', 'smoke', 'drink', 'background_mother', 'background_father', 
             'pesticide', 'has_piped_water', 'has_sewage_system', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation']	

# image labels and information

# image_ref = the reference of the image from its original dataset
# hash_code = specific hash code for this image in this dataset
# image_type = if the image is dermoscopic or clinical/smartphone images
# file_name = other type of file name/id/classification
# image_id = image identification
# lesion_id = lesion identification
# patient_id = patient identification
# dataset_source = original dataset source
# split = if the image will be reserved for training, validation, or testing
# fitzpatrick = Fitzpatrick skin tone labeling scale I-VI
# fitzpatrick_group = fitzpatrick group range I-II, III-IV, V-VI
# skin_tone = other skin tone labeling
# tone_provenance = whether the tone was computated or taken from the dataset
# sex = sex/gender
# age = raw variables, some are approximations
# anatomical_site = location of lesion
# size = size of lesion, calculated by L x W, diameter, or questionnaire
# disease_label = original disease label
# label_class = label mapped to our 8 classes
# disease_ classification = whether the lesion is benign/malignant
# pathology_confirmation = boolean for if the image diagnosis was confirmed or not
# pathology = pathology confirmation

########################################################################

# questionnaire labels 
# race
# ethnicity
# family_history
# personal_history
# cancer_history
# smoke
# drink
# background_father
# background_mother
# pesticide
# has_piped_water
# has_sewage_system
# itch
# grew
# hurt
# changed
# bleed
# elevation

In [5]:
# ISIC 2019
# Original metadata: 
# in the metadata files: image, age_approx, anatom_site_general, lesion_id, sex
# in the ground truth files: MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK, score_weight, validation_weight																															
ISIC_LABEL_COLS = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC", "UNK"]
# coarse facet derived from the raw label
ISIC_CLASSIFICATION = {
    "MEL": "malignant", "SCC": "malignant", "BCC": "malignant",
    "AK": "premalignant",
    "BKL": "benign", "NV": "benign", "DF": "benign", "VASC": "benign",
    "UNK": "unknown"}		

def load_isic_pair(gt_file, meta_file, archive) -> pd.DataFrame:
    gt = pd.read_csv(f"{META_DIR}/{gt_file}")
    meta = pd.read_csv(f"{META_DIR}/{meta_file}")
    df = gt.merge(meta, on="image", how="left", validate="one_to_one")

    onehot = df[ISIC_LABEL_COLS].fillna(0)
    raw = onehot.idxmax(axis=1).where(onehot.sum(axis=1) == 1)  # NA if no 1s
    df["disease_label"] = raw
    df["label_class"] = raw.replace({"UNK": pd.NA})
    df["disease_classification"] = (raw.map(ISIC_CLASSIFICATION).map(CLASSIFICATION_MAP))   
    df["image_ref"] = archive + "::" + Path(archive).stem + "/" + df["image"] + ".jpg"
    df["file_name"] = df["image"] + ".jpg"
    df["image_id"] = df["image"]
    df["lesion_id"] = df["lesion_id"] if "lesion_id" in df.columns else pd.NA
    site = df["anatom_site_general"].map(normalize_site)
    unseen = set(site.dropna().unique()) - set(GENERAL_SITE_MAP)
    if unseen:
        raise ValueError(f"Locations missing from GENERAL_SITE_MAP: {sorted(unseen)}")
    df["anatomical_site"] = site.map(GENERAL_SITE_MAP)
    return df

def load_isic2019() -> pd.DataFrame:
    train = load_isic_pair("ISIC_2019_Training_GroundTruth.csv", "ISIC_2019_Training_Metadata.csv", "Images/ISIC2019/ISIC_2019_Training_Input.zip")
    test = load_isic_pair("ISIC_2019_Test_GroundTruth.csv", "ISIC_2019_Test_Metadata.csv", "Images/ISIC2019/ISIC_2019_Test_Input.zip")
    df = pd.concat([train, test], ignore_index=True)
    assert len(df) == 33569, f"expected 33,569 rows, got {len(df)}"
    
    return pd.DataFrame({
        "image_ref": df["image_ref"],
        "hash_code": pd.NA, # Will be filled later with a later pass: md5 of image bytes read from the zip [TO-DO]
        "image_type": "dermoscopic",
        "file_name": df["file_name"],
        "image_id": df['image_id'],
        "patient_id": pd.NA,
        "lesion_id": df['lesion_id'],
        "dataset_source": "isic2019",
        "split": "train",
        "fitzpatrick": pd.NA,
        "fitzpatrick_group": pd.NA,
        "skin_tone": pd.NA, # Will be computed later using derm_ita [TO-DO]
        "tone_provenance": "none", # Will be filled once derm-ita is complete [TO-DO]
        "sex": df['sex'],
        "age": df['age_approx'], # Note: This is an age approximation, not an exact age
        "anatomical_site": df["anatomical_site"],
        "size": pd.NA,
        "disease_label": df["disease_label"], 
        "label_class": df["label_class"],
        "disease_classification": df["disease_classification"],
        "pathology_confirmation": pd.NA,
        "pathology": pd.NA,
        "race": pd.NA,
        "ethnicity": pd.NA,
        "family_history": pd.NA,
        "personal_history": pd.NA,
        "cancer_history": pd.NA,
        "smoke": pd.NA,
        "drink": pd.NA,
        "background_mother": pd.NA,
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA,
        "has_sewage_system": pd.NA,
        "itch": pd.NA,
        "grew": pd.NA,
        "hurt": pd.NA,
        "changed": pd.NA,
        "bleed": pd.NA,
        "elevation": pd.NA,
    })

In [6]:
# MILK10k
# Original metadata: 
# In milk10k file: isic_id, attribution, copyright_license, age_approx, anatom_site_general, anatom_site_special, concomitant_biopsy, diagnosis_1, diagnosis_2, diagnosis_3, diagnosis_4, diagnosis_confirm_type, image_manipulation, image_type, lesion_id, melanocytic, sex																														
# In training_input file: lesion_id, image_type, isic_id, attribution, copyright_license, image_manipulation, age_approx, sex	skin_tone_class, site, MONET_ulceration_crust, MONET_hair, MONET_vasculature_vessels, MONET_erythema, MONET_pigmented, MONET_gel_water_drop_fluid_dermoscopy_liquid, MONET_skin_markings_pen_ink_purple_pen
MILK_LABEL_MAP = {
    # MEL
    "Melanoma Invasive": "MEL", "Melanoma in situ": "MEL", 
    "Melanoma metastasis": "MEL", "Melanoma, NOS": "MEL", 
    # BCC / SCC
    "Basal cell carcinoma": "BCC", 
    "Squamous cell carcinoma in situ": "SCC", "Squamous cell carcinoma, Invasive": "SCC", "Squamous cell carcinoma, NOS": "SCC",
    # AK / NV / DF
    "Solar or actinic keratosis": "AK", 
    "Nevus": "NV", 
    "Dermatofibroma": "DF",
    # BKL family (per ISIC's BKL definition)
    "Solar lentigo": "BKL", "Seborrheic keratosis": "BKL",
    "Lichen planus like keratosis": "BKL", "Ink-spot lentigo": "BKL",
    # VASC family (per ISIC's VASC definition)
    "Hemangioma": "VASC", "Angiokeratoma": "VASC", "Pyogenic granuloma": "VASC",
    # Keratoacanthoma — judgment call: SCC (KA as well-differentiated SCC)
    "Keratoacanthoma": "SCC",
    # No ISIC home — excluded from training, kept in manifest
    "Clear cell acanthoma": pd.NA, 
    "Infundibular or epidermal cyst": pd.NA,
    "Juvenile xanthogranuloma": pd.NA, 
    "Mastocytosis": pd.NA, 
    "Molluscum": pd.NA,
    "Mucosal melanotic macule": pd.NA, 
    "Porokeratosis": pd.NA,
    "Sebaceous hyperplasia": pd.NA, 
    "Supernumerary nipple": pd.NA,
    "Trichoblastoma": pd.NA}

MILK_IMPUT_COLS = ["skin_tone_class", "site"]

def load_milk10k_pair(meta_file, imput_file, archive) -> pd.DataFrame:
    meta = pd.read_csv(f"{META_DIR}/{meta_file}")
    imput = pd.read_csv(f"{META_DIR}/{imput_file}")

    # Check the columns needed from each file.
    required_meta = {"isic_id", "diagnosis_3"}
    required_imput = {"isic_id", *MILK_IMPUT_COLS}

    missing_meta = required_meta - set(meta.columns)
    missing_imput = required_imput - set(imput.columns)

    if missing_meta:
        raise ValueError(f"{meta_file}: missing columns {sorted(missing_meta)}")
    if missing_imput:
        raise ValueError(f"{imput_file}: missing columns {sorted(missing_imput)}")

    if meta["isic_id"].isna().any() or imput["isic_id"].isna().any():
        raise ValueError("Missing isic_id values; inspect the source files.")

    # Only bring in the additional fields, not duplicate demographic columns.
    imput = imput[["isic_id", *MILK_IMPUT_COLS]].copy()

    df = meta.merge(imput,on="isic_id",how="left",validate="one_to_one",indicator=True,)

    df = df.drop(columns="_merge")

    df["file_name"] = df["isic_id"].astype("string") + ".jpg"
    df["image_ref"] = archive + "::milk10/" + df["file_name"]

    return df
																																							
def load_milk10k() -> pd.DataFrame:
    df = load_milk10k_pair("milk10k.csv","milk10k_training_input.csv", "Images/milk10k.zip",)

    # Check dictionary membership, not whether the mapped result is missing.
    # Some diagnoses intentionally map to pd.NA.
    unseen = set(df["diagnosis_3"].dropna().unique()) - set(MILK_LABEL_MAP)

    if unseen:
        raise ValueError(f"Labels missing from MILK_LABEL_MAP: {sorted(unseen)}")

    df["label_class"] = df["diagnosis_3"].map(MILK_LABEL_MAP)

    site = df["anatom_site_general"].map(normalize_site)
    unseen = set(site.dropna().unique()) - set(GENERAL_SITE_MAP)
    if unseen:
        raise ValueError(f"Locations missing from GENERAL_SITE_MAP: {sorted(unseen)}")
    df["anatomical_site"] = site.map(GENERAL_SITE_MAP)

    # Positive confirmation only where explicitly recognized.
    # Keep missing/unreviewed confirmation methods unknown.
    confirmation_method = (df["diagnosis_confirm_type"].astype("string").str.strip().str.lower().replace("", pd.NA))
    df["pathology_confirmation"] = (confirmation_method.map({"histopathology": 1}).astype("Int64"))
    
    classification = (df["diagnosis_1"].astype("string").str.strip().str.lower().replace("", pd.NA))
    unseen = (set(classification.dropna().unique()) - set(CLASSIFICATION_MAP))
    if unseen:
        raise ValueError(f"Unmapped MILK classifications: {sorted(unseen)}")
    df["disease_classification"] = classification.map(CLASSIFICATION_MAP)
    
    return pd.DataFrame({
        "image_ref": df['image_ref'],
        "hash_code": pd.NA, # Will be filled later with a later pass: md5 of image bytes read from the zip [TO-DO]
        "image_type": df['image_type'],
        "file_name": df['file_name'],
        "image_id": df['isic_id'],
        "patient_id": pd.NA,
        "lesion_id": df['lesion_id'],
        "dataset_source": "milk10k",
        "split": "train",
        "fitzpatrick": pd.NA,
        "fitzpatrick_group": pd.NA,
        "skin_tone": df['skin_tone_class'],
        "tone_provenance": np.where(df["skin_tone_class"].notna(), "ita_estimate", "none"),
        "sex": df['sex'],
        "age": df['age_approx'], # Note: This is an age approximation, not an exact age
        "anatomical_site": df["anatomical_site"],
        "size": pd.NA,
        "disease_label": df['diagnosis_3'],
        "label_class": df['label_class'],
        "disease_classification": df["disease_classification"],
        "pathology_confirmation": df["pathology_confirmation"],
        "pathology": df['diagnosis_confirm_type'],
        "race": pd.NA,
        "ethnicity": pd.NA,
        "family_history": pd.NA,
        "personal_history": pd.NA,
        "cancer_history": pd.NA,
        "smoke": pd.NA,
        "drink": pd.NA,
        "background_mother": pd.NA,
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA,
        "has_sewage_system": pd.NA,
        "itch": pd.NA,
        "grew": pd.NA,
        "hurt": pd.NA,
        "changed": pd.NA,
        "bleed": pd.NA,
        "elevation": pd.NA,
    })

In [7]:
# Fitzpatrick71k
# Original metadata: orig_img_name, new_img_name, md5hash, fitzpatrick, label, nine_partition_label, three_partition_label, qc, url, url_alphanum																																							
F17K_LABEL_MAP = {
    # MEL
    "melanoma": "MEL",
    "malignant melanoma": "MEL",
    "superficial spreading melanoma ssm": "MEL",
    "lentigo maligna": "MEL",

    # BCC
    "basal cell carcinoma": "BCC",
    "basal cell carcinoma morpheiform": "BCC",
    "solid cystic basal cell carcinoma": "BCC",

    # SCC
    "squamous cell carcinoma": "SCC",

    # AK
    "actinic keratosis": "AK",

    # BKL
    "seborrheic keratosis": "BKL",

    # NV 
    "nevocytic nevus": "NV",
    "congenital nevus": "NV",
    "halo nevus": "NV",
    "becker nevus": "NV",

    # DF
    "dermatofibroma": "DF",

    # VASC
    "pyogenic granuloma": "VASC",
    "granuloma pyogenic": "VASC",

    # NA: inflammatory / eczema / drug reactions
    "allergic contact dermatitis": pd.NA,
    "eczema": pd.NA,
    "dyshidrotic eczema": pd.NA,
    "neurodermatitis": pd.NA,
    "lichen simplex": pd.NA,
    "lichen planus": pd.NA,
    "lichen amyloidosis": pd.NA,
    "perioral dermatitis": pd.NA,
    "seborrheic dermatitis": pd.NA,
    "drug eruption": pd.NA,
    "drug induced pigmentary changes": pd.NA,
    "fixed eruptions": pd.NA,
    "factitial dermatitis": pd.NA,
    "stevens johnson syndrome": pd.NA,
    "photodermatoses": pd.NA,

    # NA: papulosquamous / acne / follicular
    "psoriasis": pd.NA,
    "pustular psoriasis": pd.NA,
    "pityriasis rosea": pd.NA,
    "pityriasis rubra pilaris": pd.NA,
    "pityriasis lichenoides chronica": pd.NA,
    "prurigo nodularis": pd.NA,
    "keratosis pilaris": pd.NA,
    "acne": pd.NA,
    "acne vulgaris": pd.NA,
    "rosacea": pd.NA,
    "rhinophyma": pd.NA,
    "folliculitis": pd.NA,
    "hidradenitis": pd.NA,
    "cheilitis": pd.NA,
    "paronychia": pd.NA,

    # NA: autoimmune / connective tissue / erythemas 
    "behcets disease": pd.NA,
    "dermatomyositis": pd.NA,
    "lupus erythematosus": pd.NA,
    "lupus subacute": pd.NA,
    "scleroderma": pd.NA,
    "scleromyxedema": pd.NA,
    "livedo reticularis": pd.NA,
    "erythema annulare centrifigum": pd.NA,
    "erythema elevatum diutinum": pd.NA,
    "erythema multiforme": pd.NA,
    "erythema nodosum": pd.NA,
    "neutrophilic dermatoses": pd.NA,
    "urticaria": pd.NA,
    "urticaria pigmentosa": pd.NA,
    "calcinosis cutis": pd.NA,
    "mucinosis": pd.NA,
    "stasis edema": pd.NA,
    "sarcoidosis": pd.NA,

    # NA: blistering diseases
    "epidermolysis bullosa": pd.NA,
    "hailey hailey disease": pd.NA,
    "acquired autoimmune bullous diseaseherpes gestationis": pd.NA,

    # NA: infections / infestations
    "lyme disease": pd.NA,
    "myiasis": pd.NA,
    "nematode infection": pd.NA,
    "pediculosis lids": pd.NA,
    "scabies": pd.NA,
    "tungiasis": pd.NA,
    "tick bite": pd.NA,

    # NA: genodermatoses / metabolic 
    "acanthosis nigricans": pd.NA,
    "acrodermatitis enteropathica": pd.NA,
    "ehlers danlos syndrome": pd.NA,
    "ichthyosis vulgaris": pd.NA,
    "incontinentia pigmenti": pd.NA,
    "neurofibromatosis": pd.NA,
    "tuberous sclerosis": pd.NA,
    "xeroderma pigmentosum": pd.NA,
    "dariers disease": pd.NA,
    "porphyria": pd.NA,

    # NA: non-melanocytic nevi / hamartomas
    "epidermal nevus": pd.NA,
    "naevus comedonicus": pd.NA,
    "nevus sebaceous of jadassohn": pd.NA,

    # NA: benign adnexal / cysts
    "milia": pd.NA,
    "mucous cyst": pd.NA,
    "pilar cyst": pd.NA,
    "pilomatricoma": pd.NA,
    "syringoma": pd.NA,
    "fordyce spots": pd.NA,
    "sebaceous hyperplasia": pd.NA,

    # NA: other benign / histiocytic
    "granuloma annulare": pd.NA,
    "necrobiosis lipoidica": pd.NA,
    "keloid": pd.NA,
    "xanthomas": pd.NA,
    "juvenile xanthogranuloma": pd.NA,
    "langerhans cell histiocytosis": pd.NA,
    "aplasia cutis": pd.NA,
    "striae": pd.NA,
    "vitiligo": pd.NA,
    "neurotic excoriations": pd.NA,
    "papilomatosis confluentes and reticulate": pd.NA,

    # NA: vascular malformations (judgment — see note)
    "lymphangioma": pd.NA,
    "port wine stain": pd.NA,
    "telangiectases": pd.NA,

    # NA: malignant, no ISIC home
    "kaposi sarcoma": pd.NA,
    "mycosis fungoides": pd.NA,

    # NA: porokeratosis (consistent with MILK10k map)
    "disseminated actinic porokeratosis": pd.NA,
    "porokeratosis actinic": pd.NA,
    "porokeratosis of mibelli": pd.NA,

    # NA: field findings / other
    "sun damaged skin": pd.NA,
}
def load_fitzpatrick17k() -> pd.DataFrame:
    df = pd.read_csv(f"{META_DIR}/fitzpatrick17k.csv")
    assert len(df) == 16577, f"expected 16577 rows, got {len(df)}" 
    
    # --- label mapping ---
    unseen = set(df["label"].dropna().unique()) - set(F17K_LABEL_MAP)
    assert not unseen, f"labels missing from F17K_LABEL_MAP: {sorted(unseen)}"
    df["label_class"] = df["label"].map(F17K_LABEL_MAP)

    # --- file references ---
    df["file_name"] = df["new_img_name"]
    paths = pd.read_csv(META_DIR / "fitzpatrick17k_zip_paths.csv").set_index("file_name")["member_path"]
    df["image_ref"] = "Images/Fitzpatrick17k.zip::" + df["file_name"].map(paths)
    
    fitzpatrick = normalize_fitzpatrick(df["fitzpatrick"], missing_codes=(-1,),)
    
    classification = (df["three_partition_label"].astype("string").str.strip().str.lower().replace("", pd.NA))
    unseen = (set(classification.dropna().unique()) - set(CLASSIFICATION_MAP))
    if unseen:
        raise ValueError(f"Unmapped Fitzpatrick classifications: {sorted(unseen)}")
    df["disease_classification"] = classification.map(CLASSIFICATION_MAP)
    
    df.loc[df["label_class"].eq("AK"), "disease_classification"] = "premalignant"
    
    return pd.DataFrame({
        "image_ref": df['image_ref'],
        "hash_code": df['md5hash'],
        "image_type": "clinical",
        "file_name": df['file_name'],
        "image_id": df['new_img_name'],
        "patient_id": pd.NA,
        "lesion_id": pd.NA,
        "dataset_source": "fitzpatrick17k",
        "split": "train",
        "fitzpatrick": fitzpatrick,
        "fitzpatrick_group": pd.NA,
        "skin_tone": pd.NA,
        "tone_provenance": np.where(fitzpatrick.notna(), "recorded", "none"),
        "sex": pd.NA,
        "age": pd.NA,
        "anatomical_site": pd.NA,
        "size": pd.NA,
        "disease_label": df['label'],
        "label_class": df['label_class'], # [TO-DO] need to print and map conditions
        "disease_classification": df["disease_classification"],
        "pathology_confirmation": pd.NA,
        "pathology": pd.NA,
        "race": pd.NA,
        "ethnicity": pd.NA,
        "family_history": pd.NA,
        "personal_history": pd.NA,
        "cancer_history": pd.NA,
        "smoke": pd.NA,
        "drink": pd.NA,
        "background_mother": pd.NA,
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA,
        "has_sewage_system": pd.NA,
        "itch": pd.NA,
        "grew": pd.NA,
        "hurt": pd.NA,
        "changed": pd.NA,
        "bleed": pd.NA,
        "elevation": pd.NA,
    })																																						


In [8]:
# PAD-UFES-20
# Original metadata: patient_id	lesion_id, smoke, drink, background_father, background_mother, age, pesticide, gender, skin_cancer_history, cancer_history,
# has_piped_water, has_sewage_system, fitzpatrick, region, diameter_1, diameter_2, diagnostic, itch, grew, hurt, changed, bleed, elevation, img_id, biopsed
PAD_UFES_20_LABEL_MAP = {
    "ACK": "AK",
    "BCC": "BCC",
    "MEL": "MEL",
    "NEV": "NV",
    "SCC": "SCC",
    "SEK": "BKL",
}
PAD_UFES_20_CLASSIFICATION = {   # harmonize vocabulary with other loaders
    "BCC": "malignant", "SCC": "malignant", "MEL": "malignant",
    "ACK": "premalignant",
    "NEV": "benign", "SEK": "benign",
}

PAD_SITE_MAP = {
    "abdomen": "ABDOMEN",
    "arm": "ARM",
    "back": "BACK",
    "chest": "CHEST",
    "ear": "EAR",
    "face": "FACE",
    "foot": "FOOT",
    "forearm": "FOREARM",
    "hand": "HAND",
    "lip": "LIP",
    "neck": "NECK",
    "nose": "NOSE",
    "scalp": "SCALP",
    "thigh": "THIGH",
}

def load_pad_ufes_20() -> pd.DataFrame:
    df = pd.read_csv(f"{META_DIR}/PAD-UFES-20.csv")
    assert len(df) == 2298, f"expected 2298 rows, got {len(df)}"

    unseen = set(df["diagnostic"].dropna().unique()) - set(PAD_UFES_20_LABEL_MAP)
    assert not unseen, f"labels missing from PAD_UFES_20_LABEL_MAP: {sorted(unseen)}"
    df["label_class"] = df["diagnostic"].map(PAD_UFES_20_LABEL_MAP)

    tone_col = "fitspatrick" if "fitspatrick" in df.columns else "fitzpatrick"
    df["size_mm"] = df[["diameter_1", "diameter_2"]].apply(pd.to_numeric, errors="raise").max(axis=1)  # larger diameter in mm
    biopsed = df["biopsed"].astype(str).str.upper().map({"TRUE": 1, "FALSE": 0})
    df["lesion_key"] = df["patient_id"].astype("string") + "_" + df["lesion_id"].astype("string")

    df["file_name"] = df["img_id"]
    df["image_ref"] = "Images/PAD-UFES-20.zip::PAD-UFES-20/" + df["file_name"]
        
    site = df["region"].map(normalize_site)
    unseen = set(site.dropna().unique()) - set(PAD_SITE_MAP)
    if unseen:
        raise ValueError(f"Locations missing from PAD_SITE_MAP: {sorted(unseen)}")
    
    df["anatomical_site"] = site.map(PAD_SITE_MAP)
    df["disease_classification"] = (df["diagnostic"].map(PAD_UFES_20_CLASSIFICATION).map(CLASSIFICATION_MAP))
    fitzpatrick = normalize_fitzpatrick(df[tone_col])
    
        
    return pd.DataFrame({
        "image_ref": df['image_ref'],
        "hash_code": pd.NA, # Will be filled later with a later pass: md5 of image bytes read from the zip [TO-DO]
        "image_type": "clinical",
        "file_name": df['file_name'],
        "image_id": df['img_id'],
        "patient_id": df['patient_id'],
        "lesion_id": df['lesion_key'],
        "dataset_source": "pad-ufes-20",
        "split": "train",
        "fitzpatrick": fitzpatrick,
        "fitzpatrick_group": pd.NA,
        "skin_tone": pd.NA,
        "tone_provenance": np.where(fitzpatrick.notna(), "recorded", "none"),
        "sex": df['gender'].str.lower(), # Gender is different than sex, this acts as a proxy variable
        "age": df['age'],
        "anatomical_site": df['anatomical_site'],
        "size": df["size_mm"],
        "disease_label": df['diagnostic'],
        "label_class": df['label_class'], 
        "disease_classification": df["disease_classification"],
        "pathology_confirmation": biopsed,
        "pathology": pd.Series(np.where(biopsed == 1, "histopathology", None), index=df.index),
        "race": pd.NA,
        "ethnicity": pd.NA,
        "family_history": pd.NA,
        "personal_history": df['skin_cancer_history'],
        "cancer_history": df['cancer_history'],
        "smoke": df['smoke'],
        "drink": df['drink'],
        "background_mother": df['background_mother'],
        "background_father": df['background_father'],
        "pesticide": df['pesticide'],
        "has_piped_water": df['has_piped_water'],
        "has_sewage_system": df['has_sewage_system'],
        "itch": df['itch'],
        "grew": df['grew'],
        "hurt": df['hurt'],
        "changed": df['changed'],
        "bleed": df['bleed'],
        "elevation": df['elevation'],
    })																																						


In [9]:
# HIBA
# Original metadata: isic_id, attribution, copyright_license, age_approx, anatom_site_general, anatom_site_special, benign_malignant, concomitant_biopsy, dermoscopic_type, diagnosis, 
# diagnosis_1, diagnosis_2, diagnosis_3, diagnosis_confirm_type, family_hx_mm, fitzpatrick_skin_type, image_type, lesion_id, patient_id, personal_hx_mm, sex

def load_hiba() -> pd.DataFrame:
    df = pd.read_csv(f"{META_DIR}/hiba-skin-lesions.csv")
    assert len(df) == 1635, f"expected 1635 rows, got {len(df)}"
    
    # --- label mapping ---
    unseen = set(df["diagnosis_3"].dropna().unique()) - set(MILK_LABEL_MAP)
    assert not unseen, f"labels missing from MILK_LABEL_MAP: {sorted(unseen)}"
    df["label_class"] = df["diagnosis_3"].map(MILK_LABEL_MAP)
    
    # --- file references ---
    df["file_name"] = df["isic_id"] + ".jpg"   # verify extension against the zip
    df["image_ref"] = "Images/hiba-skin-lesions.zip::hiba-skin-lesions/" + df["file_name"]
    
    site = df["anatom_site_general"].map(normalize_site)
    unseen = set(site.dropna().unique()) - set(GENERAL_SITE_MAP)
    if unseen:
        raise ValueError(f"Locations missing from GENERAL_SITE_MAP: {sorted(unseen)}")
    df["anatomical_site"] = site.map(GENERAL_SITE_MAP)
    
    fitzpatrick = normalize_fitzpatrick(df["fitzpatrick_skin_type"])
    
    classification = (df["benign_malignant"].astype("string").str.strip().str.lower().replace("", pd.NA))
    unseen = (set(classification.dropna().unique()) - set(CLASSIFICATION_MAP))
    if unseen:
        raise ValueError(f"Unmapped HIBA classifications: {sorted(unseen)}")
    df["disease_classification"] = classification.map(CLASSIFICATION_MAP)
    
    df.loc[df["label_class"].eq("AK"), "disease_classification"] = "premalignant"
    
    method = (df["diagnosis_confirm_type"].astype("string").str.strip().str.lower().replace("", pd.NA))
    confirmation = method.map({"histopathology": 1}).astype("Int64")
    
    return pd.DataFrame({
        "image_ref": df['image_ref'],
        "hash_code": pd.NA, # Will be filled later with a later pass: md5 of image bytes read from the zip 
        "image_type": "dermoscopic",
        "file_name": df['file_name'],
        "image_id": df['isic_id'],
        "patient_id": df['patient_id'],
        "lesion_id": df['lesion_id'],
        "dataset_source": "hiba",
        "split": "train",
        "fitzpatrick": fitzpatrick,
        "fitzpatrick_group": pd.NA,
        "skin_tone": pd.NA,
        "tone_provenance": np.where(fitzpatrick.notna(), "recorded", "none"),
        "sex": df['sex'], 
        "age": df['age_approx'], # Age approximation, not exact number
        "anatomical_site": df["anatomical_site"],
        "size": pd.NA,
        "disease_label": df['diagnosis_3'],
        "label_class": df['label_class'],
        "disease_classification": df['disease_classification'],
        "pathology_confirmation": confirmation,
        "pathology": df['diagnosis_confirm_type'],
        "race": pd.NA,
        "ethnicity": pd.NA,
        "family_history": df['family_hx_mm'], # History of melanoma specifically
        "personal_history": df['personal_hx_mm'],
        "cancer_history": pd.NA,
        "smoke": pd.NA,
        "drink": pd.NA,
        "background_mother": pd.NA,
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA,
        "has_sewage_system": pd.NA,
        "itch": pd.NA,
        "grew": pd.NA,
        "hurt": pd.NA,
        "changed": pd.NA,
        "bleed": pd.NA,
        "elevation": pd.NA,
    })																																						


In [10]:
# MRA-MIDAS
# Original metadata: _unnamed_var, midas_record_id, midas_file_name, midas_iscontrol, midas_distance, midas_location, midas_path, midas_pathreport, midas_gender, midas_age,	
# midas_fitzpatrick, midas_melanoma, midas_ethnicity, midas_race, clinical_impression_1, clinical_impression_2, clinical_impression_3, length__mm_, width__mm_																													

MRA_IMAGE_TYPE_MAP = {
    "dscope": "dermoscopic",
    "6in": "clinical",
    "1ft": "clinical",
    "n/a - virtual": "clinical",
}
MRA_LABEL_MAP = {
    # DF
    "benign-dermatofibroma": "DF",
    # VASC
    "benign-hemangioma": "VASC",
    # NV
    "benign-melanocytic nevus": "NV",
    # BKL 
    "benign-seborrheic keratosis": "BKL",
    # AK 
    "malignant- ak": "AK",
    # BCC 
    "malignant- bcc": "BCC",
    # MEL
    "malignant- melanoma": "MEL",
    # SCC (invasive and in situ, consistent with the MILK10k map)
    "malignant- scc": "SCC",
    "malignant- sccis": "SCC",
    # NA: unlabeled / no ISIC home / ambiguous
    "0": pd.NA, # no label recorded
    "benign-fibrous papule": pd.NA, # benign, not an ISIC class
    "benign-other": pd.NA,
    "malignant- other": pd.NA,
    "melanocytic tumor, possible re-excision (severe, spitz, aimp)": pd.NA,
    "other- melanocytic lesion, possible re-excision (severe, spitz, aimp)": pd.NA,
    "other- non-neoplastic, inflammatory, infectious": pd.NA,
}

def load_mra_midas() -> pd.DataFrame:
    df = pd.read_csv(META_DIR / "mra-midas-corrected.csv")

    # --- controls: no pathology, no label ---
    controls = df["midas_iscontrol"].astype(str).str.upper().eq("TRUE")
    if df.loc[controls, "midas_path"].notna().any():
        raise ValueError("control rows with a path label found — check midas_iscontrol")
    df = df[~controls].copy()

    df["midas_path"] = df["midas_path"].replace("0", pd.NA)

    # --- label split from midas_path ("malignant- ak" -> ("malignant", "ak")) ---
    path = (df["midas_path"].astype("string").str.strip().str.lower().replace({"": pd.NA, "0": pd.NA}))

    unseen = set(path.dropna().unique()) - set(MRA_LABEL_MAP)
    if unseen:
        raise ValueError(f"Labels missing from MRA_LABEL_MAP: {sorted(unseen)}")

    df["label_class"] = path.map(MRA_LABEL_MAP)

    # Preserve the complete source diagnosis rather than splitting text that
    # may contain an internal hyphen, such as "re-excision".
    df["disease_label"] = df["midas_path"]

    source_classification = path.str.extract(r"^(benign|malignant|other)(?:\s*-|$)", expand=False)
    df["disease_classification"] = (source_classification.map(CLASSIFICATION_MAP))
    
    # Changing Actinic Keratosis to premalignant to standardize across datasets
    df.loc[df["label_class"].eq("AK"), "disease_classification"] = "premalignant"

    # --- image type from midas_distance ---
    df["image_type"] = (df["midas_distance"].str.strip().str.lower().map(MRA_IMAGE_TYPE_MAP))

    # --- fitzpatrick: extract the numeral from "ii fair skin, blue eyes" ---
    df["fitz_raw"] = (df["midas_fitzpatrick"].str.strip().str.lower().str.extract(r"^(i{1,3}|iv|v|vi)\b", expand=False))
    unparsed = df["midas_fitzpatrick"].notna() & df["fitz_raw"].isna()
    if unparsed.any():
        raise ValueError(f"MRA Fitzpatrick not parsed: {df.loc[unparsed, 'midas_fitzpatrick'].unique().tolist()}")

    # --- size: larger of length/width, diameter in mm---
    df["size_mm"] = df[["length__mm_", "width__mm_"]].apply(pd.to_numeric, errors="raise").max(axis=1)  # larger diameter in mm

    # --- file references ---
    df["file_name"] = df["midas_file_name"]
    df["image_ref"] = "Images/MRA-MIDAS.zip::MRA-MIDAS/" + df["file_name"]
    
    assert len(df) > 0, "no rows left after removing controls"
    # set the expected count after your first run; controls removed
    # assert len(df) == EXPECTED, f"expected {EXPECTED} rows, got {len(df)}"
    fitzpatrick = normalize_fitzpatrick(df["fitz_raw"])
    
    site = df["midas_location"].map(normalize_site)
    unseen = set(site.dropna().unique()) - set(MRA_SITE_MAP)
    if unseen:
        raise ValueError(f"Locations missing from MRA_SITE_MAP: {sorted(unseen)}")
    df["anatomical_site"] = site.map(MRA_SITE_MAP)
    
    confirmation = df["midas_path"].notna().astype("Int64")

    frame = pd.DataFrame({
        "image_ref": df['image_ref'],
        "hash_code": pd.NA, # Will be filled later with a later pass: md5 of image bytes read from the zip [TO-DO]
        "image_type": df['image_type'],
        "file_name": df['file_name'],
        "image_id": df['midas_record_id'],
        "patient_id": pd.NA,
        "lesion_id": pd.NA,
        "dataset_source": "mra_midas",
        "split": "external_test",        
        "fitzpatrick": fitzpatrick,
        "fitzpatrick_group": pd.NA,
        "skin_tone": pd.NA,
        "tone_provenance": np.where(fitzpatrick.notna(), "recorded", "none"),
        "sex": df['midas_gender'].str.lower(),
        "age": df['midas_age'],
        "anatomical_site": df['anatomical_site'],
        "size": df['size_mm'],
        "disease_label": df['disease_label'],
        "label_class": df["label_class"],
        "disease_classification": df['disease_classification'],
        "pathology_confirmation": confirmation, # every non-control row has a path report
        "pathology": "histopathology",
        "race": df['midas_race'],
        "ethnicity": df['midas_ethnicity'],
        "family_history": pd.NA,
        "personal_history": pd.NA,
        "cancer_history": pd.NA,
        "smoke": pd.NA, 
        "drink": pd.NA,
        "background_mother": pd.NA, 
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA, 
        "has_sewage_system": pd.NA,
        "itch": pd.NA, 
        "grew": pd.NA, 
        "hurt": pd.NA, 
        "changed": pd.NA,
        "bleed": pd.NA, 
        "elevation": pd.NA,
    })

    exclusions = pd.read_csv(META_DIR / "mra_exclusions.csv")
    excluded = frame["image_ref"].isin(exclusions["image_ref"])

    print(f"MRA: {len(frame)} before exclusions; "f"{int(excluded.sum())} excluded; "f"{int((~excluded).sum())} retained")
    return frame.loc[~excluded].copy()

In [11]:
# DDI
# Original metadata: _unnamed_var, DDI_ID, DDI_file, skin_tone, malignant, disease

DDI_LABEL_MAP = {
    # MEL
    "melanoma": "MEL", "melanoma-acral-lentiginous": "MEL",
    "melanoma-in-situ": "MEL", "nodular-melanoma-(nm)": "MEL",
    # BCC
    "basal-cell-carcinoma": "BCC", "basal-cell-carcinoma-nodular": "BCC",
    "basal-cell-carcinoma-superficial": "BCC",
    # SCC (KA grouped with SCC, consistent with MILK10k)
    "squamous-cell-carcinoma": "SCC", "squamous-cell-carcinoma-in-situ": "SCC",
    "squamous-cell-carcinoma-keratoacanthoma": "SCC",
    # AK
    "actinic-keratosis": "AK",
    # BKL family
    "benign-keratosis": "BKL", "lichenoid-keratosis": "BKL",
    "inverted-follicular-keratosis": "BKL", "solar-lentigo": "BKL",
    "seborrheic-keratosis": "BKL", "seborrheic-keratosis-irritated": "BKL",
    # NV
    "melanocytic-nevi": "NV", "dysplastic-nevus": "NV", "blue-nevus": "NV",
    # DF
    "dermatofibroma": "DF",
    # VASC
    "angioma": "VASC", "arteriovenous-hemangioma": "VASC",
    "pyogenic-granuloma": "VASC", "hematoma": "VASC",

    # judgment calls
    "verruca-vulgaris": pd.NA, # Sometimes categorized as BKL
    "wart": pd.NA, # Sometimes categorized as BKL
    "xanthogranuloma": pd.NA, # Sometimes categorized as DF

    # NA: inflammatory / infectious
    "eczema-spongiotic-dermatitis": pd.NA,
    "morphea": pd.NA,
    "graft-vs-host-disease": pd.NA,
    "prurigo-nodularis": pd.NA, 
    "folliculitis": pd.NA,
    "abscess": pd.NA,
    "coccidioidomycosis": pd.NA,
    "molluscum-contagiosum": pd.NA,
    "onychomycosis": pd.NA,
    "tinea-pedis": pd.NA,
    "condyloma-accuminatum": pd.NA,
    # NA: benign adnexal / cystic / fibrous
    "acquired-digital-fibrokeratoma": pd.NA,
    "acrochordon": pd.NA,
    "angioleiomyoma": pd.NA,
    "blastic-plasmacytoid-dendritic-cell-neoplasm":pd.NA,
    "cellular-neurothekeoma": pd.NA,
    "chondroid-syringoma": pd.NA,
    "clear-cell-acanthoma": pd.NA,
    "dermatomyositis": pd.NA,
    "eccrine-poroma": pd.NA,
    "epidermal-cyst": pd.NA,
    "epidermal-nevus": pd.NA,
    "fibrous-papule": pd.NA, 
    "focal-acral-hyperkeratosis": pd.NA,
    "foreign-body-granuloma": pd.NA,
    "glomangioma": pd.NA,
    "hyperpigmentation": pd.NA,
    "keloid": pd.NA,
    "lipoma": pd.NA,
    "lymphocytic-infiltrations": pd.NA,
    "nevus-lipomatosus-superficialis": pd.NA, 
    "neurofibroma": pd.NA,
    "neuroma": pd.NA,
    "reactive-lymphoid-hyperplasia": pd.NA,
    "scar": pd.NA,
    "syringocystadenoma-papilliferum": pd.NA, 
    "trichilemmoma": pd.NA,
    "trichofolliculoma": pd.NA,
    "verruciform-xanthoma": pd.NA,
    # NA: malignant, no ISIC home
    "kaposi-sarcoma": pd.NA, 
    "leukemia-cutis": pd.NA,
    "metastatic-carcinoma": pd.NA, 
    "mycosis-fungoides": pd.NA,
    "sebaceous-carcinoma": pd.NA, 
    "subcutaneous-t-cell-lymphoma": pd.NA,
    # NA: ambiguous melanocytic (Spitzoid)
    "atypical-spindle-cell-nevus-of-reed": pd.NA,
    "pigmented-spindle-cell-nevus-of-reed": pd.NA,
    # NA: other
    "abrasions-ulcerations-and-physical-injuries": pd.NA,
    "acne-cystic": pd.NA, 
    "acral-melanotic-macule": pd.NA,
}

def load_ddi() -> pd.DataFrame:
    df = pd.read_csv(f"{META_DIR}/ddi_metadata.csv")
    assert len(df) == 656, f"expected 656 rows, got {len(df)}" 

    # --- label mapping ---
    unseen = set(df["disease"].dropna().unique()) - set(DDI_LABEL_MAP)
    assert not unseen, f"labels missing from DDI_LABEL_MAP: {sorted(unseen)}"
    df["label_class"] = df["disease"].map(DDI_LABEL_MAP)

    # Normalize the source's binary flag without treating missing/invalid
    # values as benign.
    malignant_text = (df["malignant"].astype("string").str.strip().str.lower().replace("", pd.NA))

    MALIGNANT_FLAG_MAP = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    unexpected = (set(malignant_text.dropna().unique())- set(MALIGNANT_FLAG_MAP))

    if unexpected:
        raise ValueError(f"Unexpected DDI malignant values: {sorted(unexpected)}")

    malignant_flag = (malignant_text.map(MALIGNANT_FLAG_MAP).astype("boolean"))

    # AK is deliberately excluded from this check.
    expected_malignant = df["label_class"].isin(["MEL", "BCC", "SCC"])
    expected_benign = df["label_class"].isin(["NV", "BKL", "DF", "VASC"])

    contradiction = ((expected_malignant & malignant_flag.eq(False)) | (expected_benign & malignant_flag.eq(True))).fillna(False)
    
    if contradiction.any():
        display(df.loc[contradiction,["DDI_ID", "disease", "label_class", "malignant"],])
        raise ValueError(f"{contradiction.sum()} DDI rows need label/flag review.")

    # Preserve the source's binary classification.
    # Missing flags stay missing.
    df["disease_classification"] = malignant_flag.map({True: "malignant", False: "benign"}).astype("string")
    
    df.loc[df["label_class"].eq("AK"), "disease_classification"] = "premalignant"

    # DDI provides grouped types, not individual Fitzpatrick types.
    DDI_TONE_MAP = {
        12: "I-II",
        34: "III-IV",
        56: "V-VI" }
    tone_codes = pd.to_numeric(df["skin_tone"], errors="raise")

    unexpected = set(tone_codes.dropna().unique()) - set(DDI_TONE_MAP)
    if unexpected:
        raise ValueError(f"Unexpected DDI skin-tone codes: {unexpected}")

    df["fitzpatrick_group"] = tone_codes.map(DDI_TONE_MAP).astype("string")

    # file references
    df["file_name"] = df["DDI_file"]
    df["image_ref"] = "Images/DDI.zip::" + df["file_name"]

    return pd.DataFrame({
        "image_ref": df["image_ref"],
        "hash_code": pd.NA,                
        "image_type": "clinical",
        "file_name": df["file_name"],
        "image_id": df["DDI_ID"],
        "patient_id": pd.NA,
        "lesion_id": pd.NA,
        "dataset_source": "ddi",
        "split": "external_test",           
        "fitzpatrick": pd.NA,
        "fitzpatrick_group": df["fitzpatrick_group"],
        "skin_tone": pd.NA,
        "tone_provenance": np.where(df["fitzpatrick_group"].notna(), "recorded", "none"),
        "sex": pd.NA,                  
        "age": pd.NA,
        "anatomical_site": pd.NA,
        "size": pd.NA,
        "disease_label": df["disease"],
        "label_class": df["label_class"],
        "disease_classification": df["disease_classification"],
        "pathology_confirmation": pd.NA,       
        "pathology": pd.NA,
        "race": pd.NA, 
        "ethnicity": pd.NA,
        "family_history": pd.NA, 
        "personal_history": pd.NA, 
        "cancer_history": pd.NA,
        "smoke": pd.NA, 
        "drink": pd.NA,
        "background_mother": pd.NA, 
        "background_father": pd.NA,
        "pesticide": pd.NA,
        "has_piped_water": pd.NA, 
        "has_sewage_system": pd.NA,
        "itch": pd.NA, 
        "grew": pd.NA, 
        "hurt": pd.NA, 
        "changed": pd.NA,
        "bleed": pd.NA, 
        "elevation": pd.NA,
    })

In [12]:
loader_functions = {
    "isic2019": load_isic2019,
    "milk10k": load_milk10k,
    "fitzpatrick17k": load_fitzpatrick17k,
    "pad-ufes-20": load_pad_ufes_20,
    "hiba": load_hiba,
    "mra_midas": load_mra_midas,
    "ddi": load_ddi,
}
allowed_labels = {"MEL", "BCC", "SCC", "AK", "BKL", "NV", "DF", "VASC"}
allowed_classification = {
    "benign", "malignant", "premalignant",
    "non_neoplastic", "indeterminate", "other",
}

for source, loader in loader_functions.items():
    frame = loader()
    missing = set(CANONICAL) - set(frame.columns)
    extra = set(frame.columns) - set(CANONICAL)
    if not frame.columns.is_unique or missing or extra:
        raise ValueError(
            f"{source}: schema mismatch; missing={sorted(missing)}, "
            f"extra={sorted(extra)}, unique_columns={frame.columns.is_unique}"
        )
    if set(frame["dataset_source"].dropna().unique()) != {source}:
        raise ValueError(f"{source}: incorrect dataset_source values")
    if frame["dataset_source"].isna().any():
        raise ValueError(f"{source}: missing dataset_source")
    for column, allowed in [
        ("label_class", allowed_labels),
        ("disease_classification", allowed_classification),
    ]:
        unexpected = set(frame[column].dropna().unique()) - allowed
        if unexpected:
            raise ValueError(
                f"{source}: unexpected {column}: {sorted(unexpected)}"
            )
    print(source, len(frame), "schema and label vocabulary OK")

isic2019 33569 schema and label vocabulary OK
milk10k 10480 schema and label vocabulary OK
fitzpatrick17k 16577 schema and label vocabulary OK
pad-ufes-20 2298 schema and label vocabulary OK
hiba 1635 schema and label vocabulary OK
MRA: 3416 before exclusions; 17 excluded; 3399 retained
mra_midas 3399 schema and label vocabulary OK
ddi 656 schema and label vocabulary OK


In [13]:
scripts_path = str(PROJECT_ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

import build_master
build_master = importlib.reload(build_master)

loader_names = [
    "load_isic2019",
    "load_milk10k",
    "load_fitzpatrick17k",
    "load_pad_ufes_20",
    "load_hiba",
    "load_mra_midas",
    "load_ddi",
]

missing = [name for name in loader_names if not callable(globals().get(name))]
if missing:
    raise NameError(f"Run or fix these loader-definition cells first: {missing}")

notebook_module = types.ModuleType("skinai_notebook_loaders")
for name in loader_names:
    setattr(notebook_module, name, globals()[name])

sys.modules[notebook_module.__name__] = notebook_module

def run_master_build(*extra_args):
    arguments = [
        "build_master.py",
        "--loaders", notebook_module.__name__,
        "--out", str(OUTPUT_DIR),
        *map(str, extra_args),
    ]
    # Use the existing complete builder, without Jupyter's command-line arguments.
    with patch.object(sys, "argv", arguments):
        build_master.main()




In [14]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

# Include all original CSVs your loaders use, including ground truth and
# supplemental tone files. Do not include generated master/report files.
# This assumes META_DIR is dedicated to original input metadata.
input_paths = sorted(META_DIR.rglob("*.csv"))
if not input_paths:
    raise FileNotFoundError(f"No input CSVs found in {META_DIR}")

input_inventory = pd.DataFrame([
    {
        "metadata_path": path.relative_to(META_DIR).as_posix(),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in input_paths
])

input_inventory.to_csv(OUTPUT_DIR / "metadata_inputs.csv", index=False)
display(input_inventory)

,metadata_path,bytes,sha256
0,ddi_metadata.csv,30679,47884b2b1b65a685b2d6321e122333b4ecb750fc5885ad...
1,fitzpatrick17k.csv,5100437,7279d6c01f95871978b3345b9c64e504d6bb2758c40998...
2,fitzpatrick17k_zip_paths.csv,1374121,534a67cd27d07f137c8c9b76572f43ca39b68cff9bf02a...
3,hiba-skin-lesions.csv,392730,57d0b754effb4d066bc7e75f964522fa01867ebea67a63...
4,ISIC_2019_Test_GroundTruth.csv,464810,2efa18419283a388080c97a02d2341de4fcb7fd696cf8d...
5,ISIC_2019_Test_Metadata.csv,293638,1424022e92b6db4baae0b6e89bb7d4e71689a7ddaabc37...
6,ISIC_2019_Training_GroundTruth.csv,1291479,aa88e9638fe4df9ef330dc4ba22fa4bd475c44af692fff...
7,ISIC_2019_Training_Metadata.csv,1214351,d93994a8ed201d474de9a7af7e17ec30929cbc4a522065...
8,milk10k.csv,2129778,347c3a386331c1a8948f4d87a952be9aff3e859b668ddb...
9,milk10k_training_input.csv,2587003,ec9a30c4d9f56bd89d17d98d0e24624423a9cea840e80e...


In [15]:
run_master_build(
    "--expected-counts", str(OUTPUT_DIR / "expected_counts.csv"),
    "--bucket", "sarcolemma/SkinAI",
    "--zip-check", "remote",
)

Loading isic2019 ...
Loading milk10k ...
Loading fitzpatrick17k ...
Loading pad_ufes_20 ...
Loading hiba ...
Loading mra_midas ...
MRA: 3416 before exclusions; 17 excluded; 3399 retained
Loading ddi ...


C:\Users\12148\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Wrote C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2\manifests\master.parquet: 68,614 rows
Reports: C:\Users\12148\OneDrive\Documents\Capstone\Capstone460v2\manifests\validation


In [16]:
# Companion CSV copy; the parquet stays the source of truth.
master = pd.read_parquet(OUTPUT_DIR / "master.parquet")
master.to_csv(OUTPUT_DIR / "master.csv", index=False)

In [17]:
#MOVE TO SEPARATE NOTEBOOK
import zipfile
from pathlib import PurePosixPath
from huggingface_hub import HfFileSystem

fs = HfFileSystem(skip_instance_cache=True)

archives = {
    "fitzpatrick17k": "Images/Fitzpatrick17k.zip",
    "pad": "Images/PAD-UFES-20.zip",
}

for source, archive in archives.items():
    with fs.open(f"buckets/sarcolemma/SkinAI/{archive}", "rb") as stream:
        with zipfile.ZipFile(stream) as z:
            members = [
                item.filename
                for item in z.infolist()
                if not item.is_dir()
                and ".ipynb_checkpoints/" not in item.filename
                and PurePosixPath(item.filename).suffix.lower()
                in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
            ]

    lookup = pd.DataFrame({
        "file_name": [PurePosixPath(name).name for name in members],
        "member_path": members,
    })

    if lookup.empty:
        raise ValueError(f"{archive}: no image members found; inspect nesting")

    duplicates = lookup["file_name"].duplicated(keep=False)
    if duplicates.any():
        display(lookup.loc[duplicates])
        raise ValueError(f"{archive}: duplicate filenames require review")

    lookup.to_csv(META_DIR / f"{source}_zip_paths.csv", index=False)
    print(source, len(lookup), "image paths recorded")

fitzpatrick17k 16577 image paths recorded
pad 2298 image paths recorded
